In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model

In [4]:
word_index=imdb.get_word_index()

rev_word_idx= {value: key for key, value in word_index.items()}


In [5]:
model=load_model('simple_rnn_imdb.h5')
model.summary()

2026-07-01 19:45:45.441998: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3
2026-07-01 19:45:45.442073: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-07-01 19:45:45.442087: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-07-01 19:45:45.442104: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-07-01 19:45:45.442121: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (32, 500, 128)         │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (32, 128)              │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (32, 1)                │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,027 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [6]:
model.get_weights()

[array([[-0.05415418, -0.06881518, -0.0827725 , ...,  0.04786806,
         -0.05408355, -0.08934572],
        [ 0.03433616, -0.04076004, -0.00376554, ...,  0.0277794 ,
         -0.06420828, -0.01000969],
        [-0.01179499,  0.02032654, -0.04701547, ...,  0.04467824,
          0.02656917, -0.03322772],
        ...,
        [ 0.15957023, -0.20021175, -0.1092459 , ...,  0.15041575,
         -0.1589414 , -0.1677537 ],
        [ 0.11054042,  0.00315521, -0.03664562, ..., -0.08467646,
          0.04177727,  0.01753598],
        [-0.0345921 , -0.05319784, -0.2939245 , ...,  0.21478228,
         -0.30106705, -0.29640457]], dtype=float32),
 array([[-0.11143198,  0.14561094,  0.12335275, ...,  0.12149334,
         -0.0468871 , -0.11842811],
        [ 0.00643647, -0.16701542, -0.08569387, ...,  0.04489869,
          0.06347588, -0.03858026],
        [ 0.11851417, -0.09744759, -0.07431123, ...,  0.00249964,
          0.10907251, -0.05462058],
        ...,
        [-0.08139949,  0.01281093, -0.0

In [7]:
# step 2 :- helper function
# Function to decode reviews

def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

# function to preprocess user input

def preprocess_text(text):
    words=text.lower().split()
    encoded_review=[word_index.get(word,2) + 3 for word in words]
    padded_review=sequence.pad_sequences([encoded_review],maxlen=500)
    return padded_review

In [8]:
# Prediction function

def predict_sentiment(review):
    preprocessed_input=preprocess_text(review)

    prediction=model.predict(preprocessed_input)
    sentiment = 'Positive' if prediction[0][0] > 0.5 else 'Negative'
    return sentiment,prediction[0][0]

In [9]:
example_review="This movie was fantastic! The acting was great and the plot was thrilling."
sentiment,score=predict_sentiment(example_review)
print(f'Review: {example_review}')
print(f'Sentiment: {sentiment}')
print(f'Prediction Score: {score}')

2026-07-01 19:55:21.797337: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step
Review: This movie was fantastic! The acting was great and the plot was thrilling.
Sentiment: Positive
Prediction Score: 0.9857938885688782
